[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emersonnjsantos/visao_computacional/blob/main/ATIVIDADE_7/relatorio_atividade_7_projeto.ipynb)

# Atividade 7 — Análise de Imagens no Domínio da Frequência
## Unidade 2 | Capítulo 1 | Tarefa 2

**Programa:** Residência em TIC43 — Módulo de Visão Computacional  
*Parceria: Softex • Ministério da Ciência, Tecnologia e Inovações (MCTI) • Governo Federal • iRede*  
**Aluno:** Emerson Noe Jose dos Santos  
**Docente Responsável:** Prof. Alyson Bezerra  
**Docentes do Módulo:** Prof. Rafael Carmo | Prof. Matheus Araújo  
**Repositório do Projeto:** [github.com/emersonnjsantos/visao_computacional](https://github.com/emersonnjsantos/visao_computacional)  

---

### Objetivo da Atividade
Compreender os fundamentos da análise de imagens no domínio da frequência, interpretar espectros obtidos pela Transformada de Fourier e analisar criticamente o funcionamento de filtros passa-baixa, passa-alta e passa-banda utilizados no processamento digital de imagens.

O relatório responde de forma discursiva, reflexiva e aprofundada a todas as 10 questões propostas no roteiro oficial, complementando a argumentação conceitual com experimentos práticos executáveis utilizando OpenCV, NumPy e Matplotlib.

In [ ]:
# Setup e Importação das Bibliotecas Essenciais
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Configurações do Matplotlib para visualização didática
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.titleweight'] = 'bold'

# Função utilitária para calcular o espectro de magnitude centralizado
def calcular_espectro_magnitude(img_gray):
    # Converte imagem para float32 e calcula DFT bidimensional
    dft = cv2.dft(np.float32(img_gray), flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)
    magnitude = cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1])
    # Aplica escala logarítmica para visualização: 20 * log(1 + magnitude)
    magnitude_log = 20 * np.log(magnitude + 1.0)
    return dft_shift, magnitude_log

# Função para criar imagem de teste rica em geometrias e frequências
def criar_imagem_sintetica_teste(h=256, w=256):
    img = np.zeros((h, w), dtype=np.uint8)
    # Fundo degradê suave (baixa frequência)
    for y in range(h):
        img[y, :] = int(180 * (y / h))
    # Formas geométricas nítidas com bordas abruptas (alta frequência)
    cv2.rectangle(img, (40, 40), (120, 120), 240, -1)
    cv2.circle(img, (180, 160), 45, 30, -1)
    # Linhas e grades finas (frequências médias e direcionais)
    for i in range(140, 240, 10):
        cv2.line(img, (i, 40), (i, 110), 220, 1)
    return img

img_teste = criar_imagem_sintetica_teste()
dft_shift_base, espectro_base = calcular_espectro_magnitude(img_teste)

print("=" * 65)
print(" AMBIENTE DE ANÁLISE NO DOMÍNIO DA FREQUÊNCIA INICIALIZADO")
print("=" * 65)
print(f"• OpenCV: {cv2.__version__}")
print(f"• NumPy:  {np.__version__}")
print("• Dimensões da Imagem Base:", img_teste.shape)
print("=" * 65)

---
## Questão 1 — Diferença entre Domínio Espacial e Domínio da Frequência

**Enunciado:**  
*Durante o estudo do processamento de imagens, aprendemos que uma imagem pode ser analisada tanto no domínio espacial quanto no domínio da frequência. Explique, com suas próprias palavras, como você entende a diferença entre esses dois domínios. Em sua resposta, comente qual deles você considera mais intuitivo para compreender o comportamento de uma imagem e explique o motivo da sua escolha.*

### Resposta Técnica e Reflexiva:

A diferença fundamental entre esses dois domínios reside na **natureza da representação da informação visual**:

1. **No Domínio Espacial:**  
   A imagem é representada como uma matriz bidimensional I(x, y), onde cada ponto possui uma coordenada física bem definida (linha e coluna) associada a um nível de intensidade ou cor (brilho do pixel). Nesse domínio, enxergamos a cena da mesma forma que os nossos olhos: objetos com formatos geométricos, texturas delimitadas e bordas espaciais. Qualquer operação realizada aqui é estritamente **local** (como a convolução espacial por uma máscara 3x3) ou **pontual** (como ajuste de contraste ou limiarização).

2. **No Domínio da Frequência:**  
   A imagem é decomposta e representada pela **taxa de variação espacial dos níveis de cinza**. Em vez de perguntar "qual é o brilho do pixel na posição (x, y)?", o domínio da frequência pergunta: *"quais são as velocidades com que as cores e sombras mudam ao longo da imagem, em quais direções essas mudanças ocorrem e com qual amplitude energética?"*. Cada ponto no espectro de Fourier representa uma componente de onda senoidal bidimensional pura que se propaga por toda a extensão da imagem. Portanto, a informação no domínio da frequência é **global**: um único ponto no espectro carrega contribuições de pixels distribuídos por toda a matriz espacial.

### Qual domínio é mais intuitivo?
Para a percepção humana, o **domínio espacial é inegavelmente mais intuitivo**. Nascemos e evoluímos processando luz organizada espacialmente no mundo tridimensional: reconhecemos imediatamente um rosto, uma árvore ou um carro pela disposição física de seus contornos e tonalidades. Pensar em uma imagem como um somatório infinito de ondas senoidais e cossenoidais bidimensionais com amplitudes e fases distintas exige um salto de abstração matemática que não faz parte da nossa vivência visual cotidiana.

No entanto, embora o domínio espacial seja mais intuitivo para a contemplação humana, o **domínio da frequência torna-se muito mais poderoso e transparente para certas tarefas de engenharia**. Fenômenos como ruídos periódicos causados por interferência de rede elétrica (que no espaço parecem um emaranhado de listras caóticas) tornam-se simples pares de pontos simétricos no domínio da frequência, permitindo uma intervenção cirúrgica impossível de ser concebida apenas olhando para a matriz de pixels.

In [ ]:
# Demonstração Prática da Questão 1: Domínio Espacial versus Domínio da Frequência
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].imshow(img_teste, cmap='gray')
axes[0].set_title("Domínio Espacial: Matriz I(x, y)
[Visão Local e Intuitiva de Pixels e Formas]")
axes[0].axis('off')

axes[1].imshow(espectro_base, cmap='magma')
axes[1].set_title("Domínio da Frequência: Espectro de Magnitude |F(u, v)|
[Visão Global: Taxa de Variação e Energia]")
axes[1].axis('off')

plt.suptitle("Comparação Visual entre os Dois Domínios de Representação da Imagem", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 2 — Explicando a Transformada de Fourier e Analogia Didática

**Enunciado:**  
*A Transformada de Fourier foi apresentada como uma ferramenta fundamental para converter uma imagem do domínio espacial para o domínio da frequência. Descreva como você explicaria esse conceito para um colega que ainda não estudou processamento de imagens. Procure usar uma analogia ou exemplo simples que ajude a entender a ideia de decompor uma imagem em frequências.*

### Resposta Didática e Conceitual:

Para explicar a Transformada de Fourier para um colega sem formação prévia na área, utilizo a **Analogia do Acorde Musical ao Piano**:

> *"Imagine que um pianista aperta três teclas ao mesmo tempo para tocar um acorde — por exemplo, as notas Dó, Mi e Sol. Quando o som chega ao seu ouvido ou a um microfone, o que existe no ar é uma única onda sonora combinada, complexa e ondulada. Se você olhar o gráfico dessa onda no computador (no domínio do tempo), verá apenas um traço oscilante sem conseguir distinguir facilmente as notas separadas.*  
>  
> *Agora, imagine que um músico com ouvido absoluto escuta esse acorde e diz com precisão imediata: 'Aqui temos uma nota Dó a 261 Hz com intensidade média, uma nota Mi a 329 Hz e uma nota Sol a 392 Hz'. Esse músico realizou exatamente o papel da **Transformada de Fourier**: ele pegou um sinal misturado e o decompôs nas notas puras (senoides) que o formaram.*  
>  
> *Na imagem digital acontece exatamente a mesma coisa, só que em duas dimensões. A imagem que você vê na tela é como o acorde final: uma mistura de milhões de variações de cor e luz. A Transformada de Fourier atua como esse músico de ouvido absoluto, separando a imagem nas suas 'notas visuais':*  
> * As **notas graves (baixas frequências)** são as transições lentas e suaves: o céu azul gradiente, a cor uniforme de uma parede, a iluminação geral.  
> * As **notas agudas (altas frequências)** são os sons estridentes: as bordas nítidas dos objetos, os fios de cabelo, as texturas ásperas e os detalhes finos.*  
>  
> *A beleza da Transformada de Fourier é que ela nos entrega a 'partitura' completa da imagem, permitindo alterar ou silenciar qualquer nota isoladamente e, depois, reconstruir a imagem perfeita através da Transformada Inversa."*

In [ ]:
# Demonstração Prática da Questão 2: Decompondo Ondas Visuais Puras (Senoides 2D)
x = np.linspace(0, 4 * np.pi, 256)
y = np.linspace(0, 4 * np.pi, 256)
X, Y = np.meshgrid(x, y)

# Onda de baixa frequência (nota grave visual)
onda_baixa = np.sin(X * 0.5)
# Onda de alta frequência (nota aguda visual)
onda_alta = np.sin(X * 4.0 + Y * 4.0)
# Combinação (o 'acorde' visual)
acorde_visual = onda_baixa + onda_alta

_, mag_baixa = calcular_espectro_magnitude((onda_baixa * 127 + 128).astype(np.uint8))
_, mag_alta = calcular_espectro_magnitude((onda_alta * 127 + 128).astype(np.uint8))

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].imshow(onda_baixa, cmap='gray')
axes[0, 0].set_title("Onda de Baixa Frequência (Transição Suave)")
axes[0, 0].axis('off')

axes[0, 1].imshow(mag_baixa, cmap='magma')
axes[0, 1].set_title("Espectro: Ponto Próximo ao Centro (Baixa Frequência)")
axes[0, 1].axis('off')

axes[1, 0].imshow(onda_alta, cmap='gray')
axes[1, 0].set_title("Onda de Alta Frequência (Variação Rápida)")
axes[1, 0].axis('off')

axes[1, 1].imshow(mag_alta, cmap='magma')
axes[1, 1].set_title("Espectro: Pontos Afastados do Centro (Alta Frequência)")
axes[1, 1].axis('off')

plt.suptitle("A Analogia do Piano na Prática: Ondas Puras e Suas Assinaturas no Espectro de Fourier", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 3 — A Região Central Brilhante do Espectro em Imagens Naturais

**Enunciado:**  
*Ao observar o espectro de frequência de uma imagem natural, normalmente encontramos uma região central bastante brilhante. Explique o que essa região representa e discuta por que muitas imagens do mundo real apresentam maior concentração de energia nessa região do espectro.*

### Resposta Técnica e Física:

Quando centralizamos o espectro de Fourier (utilizando a função `fftshift`), a frequência zero — denominada **componente DC** (Direct Current) — é posicionada no centro exato da matriz: coordenadas (M/2, N/2).

1. **O que a região central representa:**  
   * O ponto central exato F(0, 0) representa o **valor médio de intensidade de todos os pixels da imagem** multiplicado pelo tamanho da matriz. É a luminosidade global média da cena.  
   * A região circular imediatamente ao redor do centro representa as **baixas frequências espaciais**: variações lentas de brilho, transições graduais de sombras e tonalidades contínuas que cobrem grandes extensões territoriais da imagem.

2. **Por que imagens do mundo real concentram a energia no centro?**  
   Imagens naturais possuem uma propriedade estatística fundamental: **altíssima autocorrelação espacial**. No mundo real, a matéria é contínua e coerente: se você selecionar um pixel na parede de uma sala, na folha de uma árvore ou no asfalto de uma rua, a probabilidade de que os pixels vizinhos tenham uma cor e brilho quase idênticos é superior a 95%.  
   
   Superfícies homogêneas, planos de fundo, céus e corpos sólidos ocupam a esmagadora maioria da área de qualquer fotografia. Em contrapartida, as bordas abruptas (onde a cor muda repentinamente de preto para branco em apenas 1 pixel) ocupam uma fração geométrica minúscula da área da imagem.  
   
   Como a energia no domínio da frequência é proporcional à área espacial ocupada por cada padrão, a energia de qualquer imagem fotográfica natural obedece à clássica **Lei de Decaimento Espectral 1/f^2**: a amplitude das frequências cai drasticamente à medida que nos afastamos do centro em direção às altas frequências periféricas. Por isso, ao aplicar a escala logarítmica para visualização, o centro do espectro de qualquer foto do mundo real surge como um ponto ou núcleo intensamente luminoso.

In [ ]:
# Demonstração Prática da Questão 3: Perfil Radial de Energia do Espectro
h, w = espectro_base.shape
cy, cx = h // 2, w // 2

# Calcula a distância euclidiana de cada ponto do espectro até o centro DC
Y, X = np.ogrid[:h, :w]
dist_centro = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(np.int32)

# Média de energia por raio de distância do centro
raios = np.arange(0, min(cx, cy), 1)
energia_por_raio = [espectro_base[dist_centro == r].mean() for r in raios]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(espectro_base, cmap='magma')
axes[0].plot(cx, cy, 'co', markersize=8, label='Componente DC (Brilho Médio)')
circle = plt.Circle((cx, cy), 30, color='cyan', fill=False, linestyle='--', linewidth=2, label='Núcleo de Baixas Frequências')
axes[0].add_patch(circle)
axes[0].set_title("Espectro Centrado: Concentração Maciça de Energia no Centro")
axes[0].legend(loc='upper right')
axes[0].axis('off')

axes[1].plot(raios, energia_por_raio, color='crimson', linewidth=2.5)
axes[1].set_title("Decaimento de Energia em Função da Distância ao Centro (Frequência)")
axes[1].set_xlabel("Frequência Espacial (Distância Euclidiana ao Centro)")
axes[1].set_ylabel("Magnitude Média Logarítmica")
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.suptitle("Comprovação Empírica da Lei de Concentração de Energia em Baixas Frequências", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 4 — Impacto do Ruído no Espectro de Frequência

**Enunciado:**  
*Foi possível observar diferenças claras entre o espectro de uma imagem original e o espectro da mesma imagem após a adição de ruído. Descreva quais diferenças você considera mais marcantes entre esses espectros. Em sua resposta, explique como o ruído altera a distribuição das frequências e qual foi a sua principal impressão ao observar esse comportamento.*

### Resposta Técnica e Análise Crítica:

A adição de ruído (como o ruído gaussiano aditivo ou o ruído impulsivo sal e pimenta) produz uma transformação dramática na assinatura espectral da imagem:

1. **Diferenças Marcantes Observadas:**  
   * **Na imagem limpa original:** O espectro exibe um contraste elevadíssimo entre o centro brilhante e uma periferia escura e aveludada. Apenas raios e eixos direcionais bem definidos se estendem para fora do centro, correspondendo às bordas retas e aos contornos geométricos da cena.
   * **Na imagem contaminada por ruído:** As regiões periféricas que antes eram praticamente pretas tornam-se intensamente preenchidas por uma "neblina" granulada e luminosa de ruído. O espectro perde o contraste local e ganha uma camada uniforme de energia difusa em quase todas as frequências altas.

2. **Como o ruído altera a distribuição de frequências:**  
   No domínio espacial, o ruído se manifesta como perturbações aleatórias pixel a pixel: um pixel pode ter intensidade 120 e o seu vizinho imediato saltar repentinamente para 210 sem qualquer coerência estrutural.  
   Como uma transição brusca de intensidade em uma distância infinitesimal (1 pixel) corresponde à mais alta frequência espacial possível de ser amostrada pela matriz (o limite de Nyquist), **o ruído introduz energia massiva no espectro de altas frequências**. No caso do ruído branco gaussiano, ele possui densidade espectral uniforme: injeta energia de forma homogênea em todas as frequências da imagem.

3. **Principal Impressão e Reflexão:**  
   A principal impressão é de como o domínio da frequência torna o problema do ruído incrivelmente claro e tratável. No domínio espacial, o ruído parece uma contaminação caótica e inseparável dos objetos da cena. No domínio da frequência, percebemos com clareza cristalina que a imagem útil mora no centro (baixas frequências) e o ruído invasor reside primariamente nas bordas exteriores do espectro (altas frequências). Essa constatação fornece a justificativa matemática irrefutável para o uso de filtros passa-baixa.

In [ ]:
# Demonstração Prática da Questão 4: O Espectro com Ruído Gaussiano
np.random.seed(42)
ruido_gaussiano = np.random.normal(0, 28, img_teste.shape).astype(np.float32)
img_ruidosa = np.clip(img_teste.astype(np.float32) + ruido_gaussiano, 0, 255).astype(np.uint8)

_, mag_limpa = calcular_espectro_magnitude(img_teste)
_, mag_ruidosa = calcular_espectro_magnitude(img_ruidosa)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].imshow(img_teste, cmap='gray')
axes[0, 0].set_title("Imagem Original Limpa")
axes[0, 0].axis('off')

axes[0, 1].imshow(mag_limpa, cmap='magma')
axes[0, 1].set_title("Espectro Limpo: Periferia Escura (Altas Frequências Limpas)")
axes[0, 1].axis('off')

axes[1, 0].imshow(img_ruidosa, cmap='gray')
axes[1, 0].set_title("Imagem Contaminada com Ruído Gaussiano (Desvio = 28)")
axes[1, 0].axis('off')

axes[1, 1].imshow(mag_ruidosa, cmap='magma')
axes[1, 1].set_title("Espectro com Ruído: 'Neblina' Clara em Todas as Altas Frequências")
axes[1, 1].axis('off')

plt.suptitle("Comparativo do Espectro: Injeção Uniforme de Altas Frequências pelo Ruído", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 5 — Filtros Passa-Baixa no Domínio da Frequência e Aplicações Práticas

**Enunciado:**  
*Filtros passa-baixa são frequentemente utilizados para suavizar imagens. Explique, com base no que você estudou, como esse tipo de filtro atua no domínio da frequência. Em seguida, discuta em que situações reais você acredita que esse tipo de filtragem pode ser útil.*

### Resposta Técnica e Aplicações da Indústria:

1. **Como o Filtro Passa-Baixa atua no domínio da frequência:**  
   O processo de filtragem no domínio da frequência é fundamentado no **Teorema da Convolução**: uma convolução espacial complexa e demorada equivale a uma simples **multiplicação ponto a ponto** entre o espectro da imagem e a função de transferência do filtro:

   * **Equação de Filtragem:** G(u, v) = H(u, v) * F(u, v)  
   * onde F(u, v) é a Transformada de Fourier da imagem de entrada e H(u, v) é a função de transferência do filtro centrada.

   Um filtro passa-baixa é projetado para ter valores próximos de 1,0 (ganho unitário) na região central do espectro — correspondente a distâncias D(u, v) menores que um raio de corte D0 — e atenuar gradualmente para valores próximos de 0,0 à medida que nos afastamos do centro.  
   Ao zerar ou atenuar os coeficientes de alta frequência, o filtro elimina as rápidas flutuações de intensidade entre pixels vizinhos. Quando aplicamos a Transformada Inversa de Fourier (IDFT) para retornar ao domínio espacial, a imagem resultante torna-se suave, homogênea e livre de transições ásperas.

2. **Situações reais em que a filtragem passa-baixa é essencial:**  
   * **Diagnóstico Médico por Imagem (Raio-X e Tomografia Computadorizada):** Redução do ruído quântico de fótons (*quantum mottle*) gerado por sensores digitais em exames de baixa dosagem de radiação, permitindo medições mais limpas de órgãos e tecidos moles sem granulação ruidosa.
   * **Sistemas de Leitura de Placas (ALPR) e OCR Industrial:** Suavização prévia de áreas ruidosas de asfalto, respingos de lama e texturas de lataria antes de aplicar binarização adaptativa (como Otsu) ou detectores de borda, evitando que o algoritmo gere milhares de contornos falsos ao redor de sujeiras insignificantes.
   * **Imagens de Satélite e Sensoriamento Remoto:** Atenuação de ruídos térmicos e turbulências atmosféricas em imagens multiespectrais capturadas sob baixa luminosidade solar.

---
## Questão 6 — Comparativo entre Filtros Passa-Baixa (Ideal, Butterworth e Gaussiano)

**Enunciado:**  
*Três tipos de filtros passa-baixa foram apresentados. Na sua opinião, qual deles parece apresentar o comportamento mais equilibrado para aplicações práticas em imagens reais? Justifique sua resposta discutindo as características da transição entre frequências e os possíveis efeitos visuais na imagem filtrada.*

### Resposta Técnica e Comparativa:

Os três filtros clássicos diferem na forma de sua curva de transição H(u, v) em função da distância euclidiana D(u, v) ao centro:

1. **Filtro Passa-Baixa Ideal (ILPF — Ideal Lowpass Filter):**  
   * **Função:** H(u, v) = 1 se D(u, v) <= D0; e H(u, v) = 0 se D(u, v) > D0.  
   * **Características da transição:** Corte vertical abrupto em degrau.  
   * **Efeito visual destrutivo (Ringing):** Conforme demonstrado pela teoria de Fourier, a transformada inversa de um pulso retangular é uma função sinc bidimensional (sen(x)/x). A função sinc possui lóbulos secundários e ondulações periódicas pronunciadas. No espaço, isso gera o clássico **efeito de ringing** (anéis concêntricos ondulatórios e bordas-fantasma ao redor de qualquer objeto de contraste). **É inviável para aplicações reais de alta qualidade.**

2. **Filtro Passa-Baixa de Butterworth (BLPF):**  
   * **Função:** H(u, v) = 1 / [1 + (D(u, v) / D0)^(2n)], onde n é a ordem do filtro.  
   * **Características da transição:** Transição contínua e suave cuja inclinação é ajustada pelo parâmetro n.  
   * **Comportamento:** Para ordens baixas (n = 1 ou n = 2), ele realiza um corte suave e quase imperceptível de ringing. Se aumentarmos a ordem para n >= 4, sua curva aproxima-se do filtro ideal e o ringing ressurge.

3. **Filtro Passa-Baixa Gaussiano (GLPF — Gaussian Lowpass Filter):**  
   * **Função:** H(u, v) = exp( - D(u, v)^2 / (2 * D0^2) ).  
   * **Características da transição:** Transição perfeitamente suave, exponencial e assintótica.  
   * **Propriedade Matemática Fundamental:** A Transformada Inversa de Fourier de uma função gaussiana é **rigorosamente outra função gaussiana**. Como a função gaussiana é estritamente não oscilatória (não possui caudas senoidais ou lóbulos secundários), o filtro Gaussiano **NUNCA produz o efeito de ringing**, sob nenhuma condição!

### O Filtro Mais Equilibrado para Aplicações Práticas:
O **Filtro Passa-Baixa Gaussiano** (ou o filtro de Butterworth de ordem 2) é a escolha indiscutivelmente mais equilibrada e superior para aplicações práticas em imagens reais. Ele proporciona uma suavização completamente limpa, natural e esteticamente impecável, eliminando o ruído de alta frequência sem introduzir artefatos artificiais ou ondulações fantasma que possam enganar algoritmos de inteligência artificial ou a interpretação de operadores humanos.

In [ ]:
# Demonstração Prática da Questão 6: Comparação entre Ideal, Butterworth e Gaussiano
h, w = img_ruidosa.shape
cy, cx = h // 2, w // 2
D0 = 40.0  # Raio de frequência de corte

Y, X = np.ogrid[:h, :w]
D = np.sqrt((X - cx)**2 + (Y - cy)**2)

# 1. Filtro Ideal (ILPF)
H_ideal = np.zeros((h, w), dtype=np.float32)
H_ideal[D <= D0] = 1.0

# 2. Filtro Butterworth de ordem 2 (BLPF)
H_butter = 1.0 / (1.0 + (D / (D0 + 1e-5))**(2 * 2))

# 3. Filtro Gaussiano (GLPF)
H_gauss = np.exp(-(D**2) / (2 * (D0**2)))

# Aplicação dos filtros no espectro da imagem ruidosa
dft_shift_r = np.fft.fftshift(cv2.dft(np.float32(img_ruidosa), flags=cv2.DFT_COMPLEX_OUTPUT))

def aplicar_filtro_e_reconstruir(dft_shift, H):
    # Multiplicação ponto a ponto em ambos os canais (real e imaginário)
    filtrado_shift = np.zeros_like(dft_shift)
    filtrado_shift[:, :, 0] = dft_shift[:, :, 0] * H
    filtrado_shift[:, :, 1] = dft_shift[:, :, 1] * H
    # Desfaz a centralização e calcula IDFT
    f_ishift = np.fft.ifftshift(filtrado_shift)
    img_back = cv2.idft(f_ishift)
    img_back = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])
    return cv2.normalize(img_back, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

res_ideal = aplicar_filtro_e_reconstruir(dft_shift_r, H_ideal)
res_butter = aplicar_filtro_e_reconstruir(dft_shift_r, H_butter)
res_gauss = aplicar_filtro_e_reconstruir(dft_shift_r, H_gauss)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_ruidosa, cmap='gray')
axes[0].set_title("Imagem Ruidosa Original")
axes[0].axis('off')

axes[1].imshow(res_ideal, cmap='gray')
axes[1].set_title("Filtro Ideal (D0=40)
[Notar Ringing / Ondulações]")
axes[1].axis('off')

axes[2].imshow(res_butter, cmap='gray')
axes[2].set_title("Filtro Butterworth (n=2, D0=40)
[Transição Suave sem Ondulações]")
axes[2].axis('off')

axes[3].imshow(res_gauss, cmap='gray')
axes[3].set_title("Filtro Gaussiano (D0=40)
[Mais Equilibrado: Zero Ringing]")
axes[3].axis('off')

plt.suptitle("Comparativo Prático de Filtragem Passa-Baixa: Evidenciando o Efeito de Ringing", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 7 — Filtros Passa-Alta e Destrutividade Controlada de Baixas Frequências

**Enunciado:**  
*Filtros passa-alta são utilizados para destacar detalhes e bordas em uma imagem. Explique por que a remoção de baixas frequências pode produzir esse efeito. Em sua resposta, descreva também como você imagina que a aparência de uma imagem mudaria após a aplicação desse tipo de filtro.*

### Resposta Técnica e Perceptual:

1. **Por que a remoção de baixas frequências destaca as bordas?**  
   As componentes de baixa frequência no espectro representam os valores médios de iluminação e as transições suaves e uniformes de cinza. A componente DC central F(0, 0) é a soma total de brilho da cena.  
   Por outro lado, **bordas, ranhuras, contornos de objetos e detalhes finos são variações abruptas de intensidade que ocorrem em distâncias mínimas de pixels**. Matematicamente, essas transições rápidas são compostas por altas frequências espaciais.  
   Quando aplicamos um filtro passa-alta H_hp(u, v) = 1,0 - H_lp(u, v), o filtro zera completamente o centro do espectro e atenua as baixas frequências ao redor. Ao zerar a componente contínua DC, **remove-se toda a iluminação de fundo da imagem**. Sobram apenas os gradientes direcionais e as descontinuidades espaciais.

2. **Como a aparência visual da imagem muda após o filtro passa-alta puro?**  
   * **Predomínio de tons escuros/cinza:** Como a média global de brilho foi zerada, todas as superfícies homogêneas (paredes, céu, faces de objetos uniformes) tornam-se completamente pretas ou adquirem um tom de cinza neutro plano.
   * **Contornos incandescentes:** As bordas e contornos dos objetos aparecem desenhados como linhas finas, nítidas e brilhantes, realçando com clareza cada transição geométrica da cena.
   * **Aplicação em High-Boost / Unsharp Masking:** Como uma imagem puramente passa-alta é pouco estética para visualização humana direta, os engenheiros utilizam a técnica de **filtragem high-boost**: soma-se uma fração da imagem passa-alta de volta à imagem original. O resultado é uma imagem com iluminação natural completa, porém com bordas e texturas com contraste cirurgicamente reforçado.

In [ ]:
# Demonstração Prática da Questão 7: Filtro Passa-Alta Gaussiano e Realce High-Boost
H_gauss_hp = 1.0 - H_gauss  # Função de transferência passa-alta: H_hp = 1 - H_lp

res_passa_alta = aplicar_filtro_e_reconstruir(dft_shift_base, H_gauss_hp)

# High-Boost Filtering: Imagem Original + Ganho * Imagem Passa-Alta
ganho_boost = 1.4
res_high_boost = np.clip(img_teste.astype(np.float32) + ganho_boost * (res_passa_alta.astype(np.float32) - 128), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(img_teste, cmap='gray')
axes[0].set_title("Imagem Original")
axes[0].axis('off')

axes[1].imshow(res_passa_alta, cmap='gray')
axes[1].set_title("Passa-Alta Gaussiano Puro (D0=40)
[Fundo Zerado, Bordas e Detalhes Destacados]")
axes[1].axis('off')

axes[2].imshow(res_high_boost, cmap='gray')
axes[2].set_title("Técnica High-Boost (Original + Ganho HP)
[Contraste Local e Nitidez Reforçados]")
axes[2].axis('off')

plt.suptitle("Efeito da Remoção de Baixas Frequências e Realce de Contornos (Passa-Alta)", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 8 — Filtros Passa-Banda para Isolamento de Texturas e Aplicações

**Enunciado:**  
*Durante o estudo do domínio da frequência, discutimos que diferentes estruturas da imagem estão associadas a diferentes faixas de frequência. Com base nessa ideia, explique por que um filtro passa-banda pode ser útil para destacar determinados padrões ou texturas em uma imagem. Cite um exemplo de aplicação em que você acredita que esse tipo de filtro poderia ser interessante.*

### Resposta Técnica e Exemplo da Indústria:

1. **Por que um filtro passa-banda é útil para destacar padrões e texturas?**  
   Um filtro passa-banda (Bandpass Filter) é projetado para permitir a passagem exclusiva de uma faixa específica de frequências entre um raio inferior D_low e um raio superior D_high, atenuando tanto o que estiver abaixo quanto o que estiver acima dessa janela:

   * Ele **elimina as baixas frequências** (removendo variações lentas de iluminação geral, sombras projetadas e fundos contínuos).
   * Ele **elimina as altíssimas frequências** (removendo ruídos estocásticos de alta frequência e granulação eletrônica do sensor).
   
   Muitos padrões e texturas industriais do mundo real possuem uma **frequência espacial de repetição característica** bem definida: pense na trama de um tecido com 20 fios por milímetro, nas ranhuras de segurança de uma nota de dinheiro ou nas cristas papilares de uma impressão digital. Ao sintonizar a largura de banda do filtro na frequência fundamental daquela textura, o filtro isola seletivamente o padrão de interesse com pureza impecável, silenciando tanto a iluminação externa quanto o ruído.

2. **Exemplo de Aplicação Prática:**  
   * **Inspeção Automatizada de Tecidos na Indústria Têxtil:**  
     Em uma tecelagem automatizada, tecidos passam em alta velocidade sob câmeras de visão computacional. A textura uniforme do tecido gera picos de frequência em uma faixa específica do espectro. Se houver um defeito (um nó rompido, furo ou mancha de óleo), a periodicidade daquela faixa é alterada. Um filtro passa-banda calibrado na frequência da malha isola a textura do tecido e permite que um classificador detecte imediatamente a anomalia sem ser prejudicado pela iluminação irregular do galpão ou poeira no ambiente.

In [ ]:
# Demonstração Prática da Questão 8: Filtro Passa-Banda
D_low = 15.0
D_high = 55.0

# Criação do Filtro Passa-Banda Gaussiano: H_band = H_lp(D_high) - H_lp(D_low)
H_lp_high = np.exp(-(D**2) / (2 * (D_high**2)))
H_lp_low = np.exp(-(D**2) / (2 * (D_low**2)))
H_passa_banda = np.clip(H_lp_high - H_lp_low, 0.0, 1.0)

res_passa_banda = aplicar_filtro_e_reconstruir(dft_shift_base, H_passa_banda)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_teste, cmap='gray')
axes[0].set_title("Imagem com Múltiplas Estruturas")
axes[0].axis('off')

axes[1].imshow(H_passa_banda, cmap='magma')
axes[1].set_title(f"Máscara Passa-Banda no Espectro
[Anel: {int(D_low)} < D < {int(D_high)}]")
axes[1].axis('off')

axes[2].imshow(res_passa_banda, cmap='gray')
axes[2].set_title("Resultado Passa-Banda
[Isolamento Específico de Texturas e Traços Médios]")
axes[2].axis('off')

plt.suptitle("Aplicação de Filtro Passa-Banda para Isolamento de Faixas de Frequência", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 9 — Estratégia de Filtragem para Imagem Severamente Degradada

**Enunciado:**  
*Imagine que você recebeu uma imagem muito degradada por ruído. Descreva qual estratégia de filtragem você tentaria primeiro para melhorar essa imagem. Explique por que escolheu esse tipo de filtro e quais seriam suas expectativas em relação ao resultado final.*

### Resposta Estratégica e Racional de Engenharia:

Ao receber uma imagem severamente degradada por ruído no contexto da engenharia de visão computacional, **a primeira atitude correta não é aplicar um filtro às cegas, mas sim realizar um diagnóstico espectral**:

1. **Passo 1 — Diagnóstico no Espectro de Fourier:**  
   Calcula-se a Transformada de Fourier 2D da imagem degradada e inspeciona-se o espectro de magnitude centralizado. Este diagnóstico revela imediatamente a etiologia física do ruído:
   * **Cenário A (Ruído Periódico ou Interferência Harmônica):** O espectro exibe picos simétricos pontuais ("estrelas" brilhantes fora do centro). Esse ruído é comum em sensores alimentados por fontes com ruído elétrico de 60 Hz ou interferência eletromagnética de motores industriais.
   * **Cenário B (Ruído Estocástico / Gaussiano / Branco):** O espectro exibe uma distribuição difusa e contínua de alta energia por toda a periferia.

2. **Passo 2 — Estratégia de Filtragem Escolhida:**  
   * Se o ruído for **periódico (Cenário A)**, minha primeira escolha absoluta será um **Filtro Rejeita-Faixa Seletivo (Filtro Notch)**. O filtro Notch multiplica por zero unicamente as coordenadas pontuais onde estão os picos de frequência da interferência, preservando intacto 99% do espectro restante da imagem.
   * Se o ruído for **aleatório / gaussiano (Cenário B)**, minha escolha inicial será um **Filtro Passa-Baixa Gaussiano (GLPF)** com raio de corte D0 calibrado empiricamente (tipicamente entre 15% e 25% da dimensão menor da imagem). Escolho o Gaussiano porque ele garante a atenuação progressiva da energia de alta frequência sem introduzir os anéis espúrios de ringing do filtro ideal.

3. **Expectativas em relação ao resultado final:**  
   * **No caso do Filtro Notch:** A expectativa é de um resultado espetacular, próximo da perfeição: o padrão listrado da interferência desaparecerá completamente sem nenhum borramento nas bordas dos objetos.
   * **No caso do Filtro Passa-Baixa Gaussiano:** A expectativa é realista: haverá uma redução drástica do aspecto ruidoso e da variância espúria, tornando as superfícies lisas e estáveis. No entanto, o clássico **trade-off de engenharia (suavização versus nitidez)** se fará presente: haverá uma leve perda de definição nos detalhes mais finos da cena. Caso a nitidez seja crítica para um modelo posterior, complementarei a estratégia com uma etapa de realce de bordas (High-Boost).

In [ ]:
# Demonstração Prática da Questão 9: Diagnóstico e Remoção de Ruído Periódico com Filtro Notch
# Inserção de ruído senoidal periódico (interferência de rede/motor)
h, w = img_teste.shape
Y, X = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
frequencia_interferencia = 0.28
ruido_periodico = 45 * np.sin(2 * np.pi * frequencia_interferencia * X + 2 * np.pi * frequencia_interferencia * Y)
img_degradada_periodica = np.clip(img_teste.astype(np.float32) + ruido_periodico, 0, 255).astype(np.uint8)

# Diagnóstico no domínio da frequência
dft_deg, mag_deg = calcular_espectro_magnitude(img_degradada_periodica)

# Criação do Filtro Notch (rejeição pontual nos picos simétricos da interferência)
cy, cx = h // 2, w // 2
H_notch = np.ones((h, w), dtype=np.float32)
# Coordenadas dos picos da interferência identificados no espectro
offset_u = int(frequencia_interferencia * w)
offset_v = int(frequencia_interferencia * h)
raio_notch = 8

# Cria buracos (atenuação) nos pontos simétricos da interferência
for u0, v0 in [(cx + offset_u, cy + offset_v), (cx - offset_u, cy - offset_v)]:
    D_notch = np.sqrt((X - u0)**2 + (Y - v0)**2)
    H_notch[D_notch <= raio_notch] = 0.0

res_restaurada_notch = aplicar_filtro_e_reconstruir(dft_deg, H_notch)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_degradada_periodica, cmap='gray')
axes[0].set_title("Imagem com Severa Interferência Periódica")
axes[0].axis('off')

axes[1].imshow(mag_deg, cmap='magma')
axes[1].set_title("Diagnóstico: Picos Pontuais no Espectro
[Identificação Exata da Frequência do Ruído]")
axes[1].axis('off')

axes[2].imshow(res_restaurada_notch, cmap='gray')
axes[2].set_title("Restauração por Filtro Notch
[Interferência Eliminada, Imagem Limpa]")
axes[2].axis('off')

plt.suptitle("Estratégia de Engenharia: Diagnóstico Espectral e Restauração Cirúrgica com Filtro Notch", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Questão 10 — Reflexão Final sobre o Estudo no Domínio da Frequência

**Enunciado:**  
*Após estudar a análise de imagens no domínio da frequência, reflita sobre a seguinte questão: Qual foi o aspecto desse conteúdo que mais chamou sua atenção ou que você considerou mais interessante durante o estudo? Explique por que esse ponto foi marcante para você e como ele contribuiu para sua compreensão do processamento digital de imagem.*

### Reflexão Crítica e Maturidade de Engenharia:

O aspecto mais marcante e conceitualmente transformador de todo o estudo da análise no domínio da frequência foi a constatação da **Dualidade Espaço-Frequência através do Teorema da Convolução**:

* **Teorema da Convolução:** (f * h)(x, y) equivale no domínio da frequência a F(u, v) * H(u, v)  
* Uma operação que no domínio espacial exige o deslocamento mecânico de uma máscara sobre cada pixel da imagem (com complexidade computacional proporcional à área do kernel) transforma-se, no domínio de Fourier, em uma **simples e elegante multiplicação elemento a elemento**.

### Por que esse ponto foi marcante?
Durante os estudos iniciais de visão computacional, é comum enxergarmos filtros como pequenas janelas numéricas (kernels 3x3 ou 5x5) que deslizam sobre a matriz de pixels. Essa visão, puramente mecânica e local, muitas vezes nos impede de compreender por que um filtro funciona ou por que certas degradações ocorrem.

Ao transpor a imagem para o domínio da frequência, ocorre uma mudança profunda de perspectiva:
1. **Compreensão dos Efeitos Colaterais:** Entendi com rigor por que filtros espaciais abruptos (como a média simples) geram artefatos ondulatórios: porque a transformada inversa de um bloco retangular é a função sinc oscilatória. Isso desmistificou o fenômeno do *ringing* e me ensinou por que o filtro Gaussiano é soberano em aplicações industriais.
2. **Capacidade de Diagnóstico Cirúrgico:** A ideia de que ruídos periódicos complexos — que no espaço parecem estragar toda a cena de forma irreparável — se manifestam na frequência como pontos discretos e isolados me deu uma sensação autêntica de "superpoder de engenharia": a habilidade de enxergar a estrutura oculta dos dados e removê-la com precisão microscópica.
3. **Eficiência Computacional para Grandes Dimensões:** Em imagens modernas de alta resolução (câmeras de 12 a 60 Megapixels), aplicar convoluções espaciais com filtros grandes é extremamente custoso. Graças ao algoritmo da Transformada Rápida de Fourier (FFT), o custo computacional despenca de uma ordem quadrática para a ordem O(N * log N), viabilizando processamento em larga escala na indústria.

Essa jornada mudou definitivamente a minha mentalidade de desenvolvedor: deixei de ver o processamento digital de imagens como mero ajuste de parâmetros de biblioteca e passei a encará-lo como uma disciplina de engenharia rigorosa baseada na física dos sinais bidimensionais.

---
## Referências Bibliográficas e Fontes Consultadas

1. **GONZALEZ, Rafael C.; WOODS, Richard E.** *Digital Image Processing*. 4. ed. Nova York: Pearson, 2018. 1022 p. (Capítulo 4: Filtering in the Frequency Domain — Fundamentação clássica da DFT 2D, Teorema da Convolução, propriedades de simetria, filtros ILPF, BLPF, GLPF e filtros Notch).
2. **SZELISKI, Richard.** *Computer Vision: Algorithms and Applications*. 2. ed. Cham: Springer, 2022. 949 p. (Seção 3.2: Fourier transforms e representação de sinais bidimensionais em visão computacional).
3. **OPENCV DOCUMENTATION.** *Discrete Fourier Transform in OpenCV*. Disponível em: <https://docs.opencv.org/4.x/de/cbd/tutorial_py_fourier_transform.html>. Acesso em: 13 out. 2026.
4. **BRACEWELL, Ronald N.** *The Fourier Transform and Its Applications*. 3. ed. Boston: McGraw-Hill, 2000. 616 p. (Referência matemática rigorosa sobre teoremas de amostragem, convolução e dualidade temporal/espacial e espectral).
5. **SILVA, E. et al.** *Visão computacional na indústria: tendências e exemplos práticos*. São Paulo: Repositório Institucional da Universidade de São Paulo (USP), 2020. Disponível em: <https://repositorio.usp.br/directbitstream/e9aced0b-d8da-4956-b2e0-bd3bcbeceb24/Vis%C3%A3o%20computacional%20na%20ind%C3%BAstria%20tend%C3%AAncias%20e%20exemplos%20pr%C3%A1ticos.pdf>. Acesso em: 13 out. 2026.